## 1. Introduction

### Task: Baseline Training Notebook — BirdCLEF+ 2026

#### Research Context
- Paper: [Tackling Domain Shift in Bird Audio Classification via Transfer Learning and Semi-Supervised Distillation: A Case Study on BirdCLEF+ 2025 (Sydorskyi & Gonçalves, 2025)](https://ceur-ws.org/Vol-4038/paper_256.pdf)
- This notebook investigates transfer learning with CNN backbones (EfficientNet family) pretrained on ImageNet, fine-tuned for multi-label bioacoustic classification across 234 species (birds, amphibians, mammals, reptiles, insects) from 5-second audio chunks in BirdCLEF+ 2026.

#### Objectives
- **Compare** three EfficientNet variants on macro ROC-AUC and macro F1 (threshold = 0.5) using 3-fold cross-validation:

  | Model | timm ID |
  |---|---|
  | EfficientNetV2-S | `tf_efficientnetv2_s.in21k_ft_in1k` |
  | EfficientNetB0 | `tf_efficientnet_b0.ns_jft_in1k` |
  | EfficientNetB3 | `tf_efficientnet_b3.ns_jft_in1k` |

- **Gain insight** into whether a larger backbone (EfficientNetV2-S) justifies its additional compute cost over lighter alternatives (B0, B3) for this task
- **Establish** a stable preprocessing and baseline modelling pipeline — converting raw audio to log-Mel spectrograms (128 mel bands, 32 kHz), training with AdamW + cosine LR schedule + early stopping — as the foundation for future experiments in this project

#### Research Question
- Which EfficientNet variant achieves the best macro ROC-AUC / macro F1 trade-off as a BirdCLEF+ 2026 baseline?


## 2. Setup

In [2]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score

UTILS    = '/kaggle/input/datasets/tsapalyu/birdclef2026-utils'
COMP     = '/kaggle/input/competitions/birdclef-2026'
UPSAMPLE = '/kaggle/input/datasets/tsapalyu/birdclef2026-upsamples'

SEG_DIR  = f'{UPSAMPLE}/train_upsampled_species'   # pre-extracted 5-s clips

sys.path.append(UTILS)

from birdclef_utils.dataset import UpsampledSpeciesDataset
from birdclef_utils.constants import NUM_CLASSES

_base = {
    'batch_size':    64,
    'accum_steps':   1,
    'num_workers':   4,
    'lr':            1e-3,
    'weight_decay':  1e-5,
    'max_epochs':    20,
    'patience':      5,
    'num_classes':   NUM_CLASSES,
    'use_amp':       False,
    # loss: 1.0 * BCE + 1.0 * SigmoidFocalLoss (alpha=0.25, gamma=2)
    'bce_weight':    1.0,
    'focal_weight':  1.0,
    'focal_alpha':   0.25,
    'focal_gamma':   2.0,
}

CONFIGS = {
    'effnetv2s': {**_base,
        'model_name': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'model_key':  'effnetv2s',
    },
    'effnetb0': {**_base,
        'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
        'model_key':  'effnetb0',
    },
    'effnetb3': {**_base,
        'model_name': 'tf_efficientnet_b3.ns_jft_in1k',
        'model_key':  'effnetb3',
    },
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}")
print(f"Models to compare: {list(CONFIGS.keys())}")

Device: Tesla T4
Models to compare: ['effnetv2s', 'effnetb0', 'effnetb3']


## 3 & 4. Data Loading & Exploration and Preprocessing

In [3]:
seg_df = pd.read_csv(f'{SEG_DIR}/species_segment_index.csv')
with open(f'{UTILS}/label2idx.json') as f:
    label2idx = json.load(f)

# Verify SEG_DIR exists and a sample of segment files are reachable
assert os.path.isdir(SEG_DIR), f"SEG_DIR not found: {SEG_DIR}"
_sample_paths = seg_df['dest_path'].head(5).tolist()
_missing = [
    p for p in _sample_paths
    if not os.path.exists(os.path.join(SEG_DIR, *p.split('/')[2:]))
]
if _missing:
    print(f"WARNING: {len(_missing)}/5 sampled segment files not found in {SEG_DIR!r}:")
    for p in _missing:
        print(f"  {p}")
else:
    print(f"seg segments: {len(seg_df):,}  |  {seg_df['assigned_species'].nunique()} species  |  {NUM_CLASSES} classes  |  path check OK")

def build_loaders(fold, batch_size=32, num_workers=4):
    """Build train/val DataLoaders for one fold using UpsampledSpeciesDataset."""
    train_df = seg_df[
        (seg_df['fold'] != fold) & seg_df['assigned_species'].isin(label2idx)
    ].reset_index(drop=True)
    val_df = seg_df[
        (seg_df['fold'] == fold) & seg_df['assigned_species'].isin(label2idx)
    ].reset_index(drop=True)

    train_ds = UpsampledSpeciesDataset(train_df, SEG_DIR, label2idx, mode='train')
    val_ds   = UpsampledSpeciesDataset(val_df,   SEG_DIR, label2idx, mode='val')

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    return train_loader, val_loader, val_ds


# Sanity check: verify batch shapes for fold 0 before building models
train_loader, val_loader, val_ds = build_loaders(fold=0, batch_size=64)
specs, labels = next(iter(train_loader))
print(f"spec batch:  {specs.shape}")
print(f"label batch: {labels.shape}")
print(f"avg species per sample: {labels.sum(dim=1).mean():.2f}")

seg segments: 74,385  |  225 species  |  234 classes  |  path check OK
spec batch:  torch.Size([64, 1, 128, 313])
label batch: torch.Size([64, 234])
avg species per sample: 1.00


## 5. Model Definition

Models are built using a CNN backbone from timm with:
- ImageNet pretrained weights
- Single-channel spectrogram input (`in_chans=1`)
- 234-output multi-label classification head

Three backbones are compared:
- **EfficientNetV2-S** (`tf_efficientnetv2_s.in21k_ft_in1k`) — (as per the paper)
- **EfficientNetB0** (`tf_efficientnet_b0.ns_jft_in1k`) — lighter, faster alternative
- **EfficientNetB3** (`tf_efficientnet_b3.ns_jft_in1k`) — lighter, faster alternative

In [4]:
def build_model(cfg):
    model = timm.create_model(
        cfg['model_name'],
        pretrained=True,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    return model.to(DEVICE)

## 6. Training

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, cfg):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (specs, labels) in enumerate(loader):
        specs  = specs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # forward
        logits = model(specs)
        loss = criterion(logits, labels)

        # backward (with gradient accumulation)
        (loss / cfg['accum_steps']).backward()
        if (step + 1) % cfg['accum_steps'] == 0:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item()

    return running_loss / len(loader)


### Metrics

In [6]:
def macro_roc_auc(y_true, y_pred):
    """Mean per-class ROC-AUC, skipping classes with no positive (or no
    negative) labels — AUC is undefined for those."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        aucs.append(roc_auc_score(col, y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')


def macro_f1(y_true, y_pred, threshold=0.5):
    """Macro-averaged F1, restricted to classes that have at least one
    positive label in y_true (undefined classes are skipped, not zeroed)."""
    y_true = np.asarray(y_true)
    y_binary = (np.asarray(y_pred) >= threshold).astype(int)
    valid = y_true.sum(axis=0) > 0
    if not valid.any():
        return float('nan')
    return float(f1_score(y_true[:, valid], y_binary[:, valid],
                          average='macro', zero_division=0))

### Loss Function

Combined `1.0 · BCE + 1.0 · SigmoidFocalLoss`, following the paper (Section 5.2).

Weights and gamma are set in `_base` and can be overridden per model in `CONFIGS`.

In [10]:
class CombinedBCEFocalLoss(nn.Module):
    """1.0 · BCEWithLogitsLoss + 1.0 · torchvision SigmoidFocalLoss."""
    def __init__(self, bce_weight=1.0, focal_weight=1.0, alpha=0.25, gamma=2.0):
        super().__init__()
        self.bce_weight   = bce_weight
        self.focal_weight = focal_weight
        self.alpha        = alpha
        self.gamma        = gamma
        self.bce          = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss   = self.bce(logits, targets)
        focal_loss = torchvision.ops.sigmoid_focal_loss(
            inputs=logits,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction='mean',
        )
        return self.bce_weight * bce_loss + self.focal_weight * focal_loss


def build_criterion(cfg):
    return CombinedBCEFocalLoss(
        bce_weight=cfg.get('bce_weight', 1.0),
        focal_weight=cfg.get('focal_weight', 1.0),
        alpha=cfg.get('focal_alpha', 0.25),
        gamma=cfg.get('focal_gamma', 2.0),
    )

### Validation

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for specs, labels in loader:
        specs = specs.to(DEVICE, non_blocking=True)
        probs = torch.sigmoid(model(specs))
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.vstack(all_preds), np.vstack(all_labels)

### Fold Training

In [8]:
def train_fold(fold, cfg):
    model_key = cfg['model_key']
    print(f"\n{'='*55}\n{model_key.upper()}  —  FOLD {fold}\n{'='*55}")

    train_loader, val_loader, val_ds = build_loaders(
        fold, batch_size=cfg['batch_size'], num_workers=cfg['num_workers'],
    )

    print(f"Train samples: {len(train_loader.dataset)}")
    print(f"Val samples:   {len(val_ds)}")

    model     = build_model(cfg)
    criterion = build_criterion(cfg)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['max_epochs'],
    )

    ckpt_path = f'/kaggle/working/{model_key}_fold{fold}_last.pth'
    start_epoch = 0
    best_auc = 0.0
    epochs_no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch       = ckpt['epoch'] + 1
        best_auc          = ckpt['best_auc']
        epochs_no_improve = ckpt['epochs_no_improve']
        print(f"Resuming {model_key} fold {fold} from epoch {start_epoch} "
              f"(best AUC so far {best_auc:.4f})")
    else:
        print(f"No checkpoint found - training {model_key} fold {fold} from scratch.")

    for epoch in range(start_epoch, cfg['max_epochs']):
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     criterion, cfg)

        preds, labels = evaluate(model, val_loader)
        val_auc = macro_roc_auc(labels, preds)

        scheduler.step()

        print(f"  epoch {epoch:2d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | {time.time()-t0:.0f}s")

        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_auc':             best_auc,
            'epochs_no_improve':    epochs_no_improve,
            'val_auc':              val_auc,
            'model_name':           cfg['model_name'],
            'model_key':            model_key,
            'num_classes':          cfg['num_classes'],
            'in_chans':             1,
        }, f'/kaggle/working/{model_key}_fold{fold}_last.pth')

        if val_auc > best_auc:
            best_auc = val_auc
            epochs_no_improve = 0
            torch.save({'model_state_dict': model.state_dict(),
                        'epoch': epoch, 'val_auc': val_auc},
                       f'/kaggle/working/{model_key}_fold{fold}_best.pth')
            print(f"    -> new best ({best_auc:.4f})")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg['patience']:
            print(f"  early stopping (best val AUC {best_auc:.4f})")
            break

    return best_auc

In [12]:
# --- Training control ---
# Set MODEL_KEY to select which backbone to train.
# Set FOLD to select which fold to run.
# Run this cell once per fold per model (or loop over both if compute allows).
SMOKE_TEST = False
MODEL_KEY  = 'effnetb0'   # 'effnetv2s' | 'effnetb0' | 'effnetb3'
FOLD       = 2              # 0 | 1 | 2

cfg = CONFIGS[MODEL_KEY]

if SMOKE_TEST:
    smoke_cfg = dict(cfg, max_epochs=1)
    print(f"SMOKE TEST: one epoch, fold {FOLD}, model {MODEL_KEY}.")
    train_fold(fold=FOLD, cfg=smoke_cfg)
else:
    best = train_fold(fold=FOLD, cfg=cfg)
    print(f"\n{MODEL_KEY} fold {FOLD} best val AUC: {best:.4f}")


EFFNETB0  —  FOLD 2
Train samples: 49645
Val samples:   24740
No checkpoint found - training effnetb0 fold 2 from scratch.
  epoch  0 | loss 0.0303 | val AUC 0.9419 | 458s
    -> new best (0.9419)
  epoch  1 | loss 0.0155 | val AUC 0.9664 | 456s
    -> new best (0.9664)
  epoch  2 | loss 0.0121 | val AUC 0.9745 | 482s
    -> new best (0.9745)
  epoch  3 | loss 0.0097 | val AUC 0.9774 | 535s
    -> new best (0.9774)
  epoch  4 | loss 0.0077 | val AUC 0.9787 | 459s
    -> new best (0.9787)
  epoch  5 | loss 0.0060 | val AUC 0.9798 | 438s
    -> new best (0.9798)
  epoch  6 | loss 0.0045 | val AUC 0.9798 | 450s
  epoch  7 | loss 0.0032 | val AUC 0.9786 | 456s
  epoch  8 | loss 0.0022 | val AUC 0.9789 | 450s
  epoch  9 | loss 0.0015 | val AUC 0.9790 | 439s
  epoch 10 | loss 0.0010 | val AUC 0.9787 | 451s
  early stopping (best val AUC 0.9798)

effnetb0 fold 2 best val AUC: 0.9798


## 7. Evaluation

### Evaluation Plan
- Load best checkpoint per model per fold from saved weights
- Evaluate on held-out validation folds (3-fold CV, as per paper — Sydorskyi & Gonçalves, 2025)
- Compute macro ROC-AUC and macro F1 (threshold = 0.5)

In [13]:
def load_fold_model(checkpoint_path, cfg):
    """Rebuild a trained model from a saved checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model = timm.create_model(
        cfg['model_name'],
        pretrained=False,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    return model, ckpt

In [ ]:
CHECKPOINT_DIR = '/kaggle/input/datasets/tsapalyu/birdclef2026-baseline-weights'
NUM_FOLDS = 3

all_results = {}

for model_key, cfg in CONFIGS.items():
    fold_rows = []
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_key}  ({cfg['model_name']})")
    print(f"{'='*60}")

    for fold in range(NUM_FOLDS):
        ckpt_path = os.path.join(CHECKPOINT_DIR,
                                 f'{model_key}_fold{fold}_best.pth')
        if not os.path.exists(ckpt_path):
            print(f"  Fold {fold}: checkpoint not found, skipping.")
            continue

        print(f"\n  --- Fold {fold} ---")
        model, ckpt = load_fold_model(ckpt_path, cfg)
        print(f"  loaded epoch {ckpt['epoch']}, "
              f"training-time val AUC {ckpt['val_auc']:.4f}")

        _, val_loader, _ = build_loaders(
            fold,
            batch_size=cfg['batch_size'],
            num_workers=cfg['num_workers'],
        )

        preds, labels = evaluate(model, val_loader)
        auc = macro_roc_auc(labels, preds)
        f1  = macro_f1(labels, preds)
        print(f"  val AUC: {auc:.4f}  |  val F1 (t=0.5): {f1:.4f}")

        fold_rows.append({
            'fold':    fold,
            'val_AUC': auc,
            'val_F1':  f1,
            'n_val':   len(labels),
        })

        del model
        torch.cuda.empty_cache()

    all_results[model_key] = pd.DataFrame(fold_rows)


# --- Comparison table ---
print("\n" + "=" * 65)
print("Model Comparison — macro ROC-AUC vs macro F1 (threshold = 0.5)")
print("=" * 65)

summary_rows = []
for model_key, df in all_results.items():
    if df.empty:
        continue
    summary_rows.append({
        'model':      model_key,
        'mean_AUC':   df['val_AUC'].mean(),
        'std_AUC':    df['val_AUC'].std(ddof=1),
        'mean_F1':    df['val_F1'].mean(),
        'std_F1':     df['val_F1'].std(ddof=1),
        'folds_done': len(df),
    })

comparison_df = pd.DataFrame(summary_rows)
print(comparison_df.to_string(index=False, float_format='%.4f'))

for model_key, df in all_results.items():
    if not df.empty:
        df.to_csv(f'/kaggle/working/cv_summary_{model_key}.csv', index=False)
comparison_df.to_csv('/kaggle/working/cv_comparison.csv', index=False)
print(f"\nSaved cv_comparison.csv and per-model summaries to /kaggle/working/")

### Note: No Held-Out Test Set

BirdCLEF+ 2026 is a Kaggle competition where the test set consists of hidden soundscape recordings — participants never receive the audio files or ground-truth labels. Predictions are submitted to Kaggle's scoring server, which returns a public leaderboard score (macro ROC-AUC) computed on a subset of the hidden test data.

This means local evaluation is only possible on the labeled training data. We follow the paper (Sydorskyi & Gonçalves, 2025) and use **3-fold cross-validation** on the combined training set — focal recordings (`train.csv`: 35,549 clips, 206 species) and labeled soundscapes (`train_soundscapes_labels.csv`: 66 files, 75 species) — as a proxy for generalisation performance. The fold-level macro ROC-AUC and macro F1 reported here are the best available local signal; the true test performance is only revealed via Kaggle submission.

## 8. Comparison & Analysis

#TODO

Side-by-side comparison tables
Visualizations (bar charts, learning curves, confusion matrices)
Trade-off analysis (accuracy vs. efficiency)
Error analysis

Key Visualizations:
Learning curves (train/val loss over epochs)
Performance comparison bar charts
Confusion matrices for classification
Error case analysis

## 9. Discussion

#TODO
Interpret results

Explain unexpected findings

Discuss limitations

Connect back to research papers

Implications for MVP

## 10. References

Data source: https://www.kaggle.com/competitions/birdclef-2026/data

Confiugration settings: https://github.com/VSydorskyy/BirdCLEF_2025_2nd_place (from Reference Paper)